# **WNBA Needs Model**

By: Vedanth Chamala and Derek Yee

## **1. Select “Winning" Stats (Top 15??)**

- Use the top 15 team stats most correlated with winning (from corr.csv).

- Exclude metrics that don’t help find player type (net rating)

- Avoid redundant stats:
  - If using REB per game, do not also use REB per 100 possessions.

## **2. Create Stat Importance Weights**

- Convert correlations into importance weights
  - Weights should add up to 1
  - importance_weight = |correlation| / sum(|correlations|)


## **3. Identify Each Team’s Weaknesses**

For each team:

- Compare team performance to average

- Compute a deficit score per stat:

  - If higher is better: deficit = league_avg − team_value

  - If lower is better (turnovers): flip the sign

- Then compute:

    - Weighted deficit = deficit × importance_weight

- Each team gets a ranked list of the stats they need most.

## **4. Define Player Archetypes**


0: Support Players

1: Balanced contributors

2: Primary Creators

3: Interior Defenders

## **5. Match Team Needs to Archetype**

For each team:

- Compare the team’s weighted deficits to the strengths of each archetype

- Score each archetype by how well it addresses the team’s biggest weaknesses

Output: A ranked list of archetypes per team (best → worst fit)

In [ ]:
import os
import pandas as pd
import numpy as np

# load data
BASE_DIR = "/content/drive/MyDrive/WNBA data/"

wnba_team_data = pd.read_csv(os.path.join(BASE_DIR, "all_stats.csv"))

wnba_team_data


,Year,Team,Age,W,L,PW,PL,MOV,SOS,SRS,...,opp_FTA_per_100,opp_ORB_per_100,opp_DRB_per_100,opp_TRB_per_100,opp_AST_per_100,opp_STL_per_100,opp_BLK_per_100,opp_TOV_per_100,opp_PF_per_100,opp_PTS_per_100
0,2025,Minnesota Lynx,28.8,34,10,37,7,9.41,-0.69,8.72,...,20.1,11.7,31.8,43.4,24.5,8.7,3.7,18.3,21.3,100.0
1,2025,Washington Mystics,24.8,16,28,14,30,-4.43,0.21,-4.23,...,24.2,9.5,32.4,41.9,25.9,10.3,5.1,16.1,25.4,105.7
2,2025,Dallas Wings,25.4,10,34,11,33,-6.30,0.92,-5.38,...,27.5,10.3,33.4,43.7,29.0,9.7,5.9,16.0,22.3,111.2
3,2025,Chicago Sky,27.7,10,34,7,37,-10.05,0.92,-9.13,...,24.9,11.4,31.1,42.5,27.9,12.4,7.2,14.4,21.7,111.9
4,2025,Connecticut Sun,27.1,11,33,7,37,-10.11,0.87,-9.24,...,27.3,11.4,34.4,45.8,28.0,9.4,4.9,17.6,22.7,111.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,1997,Los Angeles Sparks,25.5,14,14,17,11,2.21,-0.28,1.94,...,29.4,14.6,27.1,41.7,20.7,13.0,3.7,22.9,26.1,91.1
361,1997,Cleveland Rockers,28.1,15,13,16,12,1.79,-0.22,1.56,...,25.6,14.7,27.0,41.7,22.3,14.6,4.6,23.7,28.8,92.0
362,1997,Charlotte Sting,27.0,15,13,16,12,1.29,-0.16,1.13,...,25.3,14.2,29.4,43.6,21.3,13.6,3.5,22.9,24.0,91.1
363,1997,Sacramento Monarchs,27.7,10,18,5,23,-7.50,0.94,-6.56,...,29.8,15.7,27.9,43.6,24.4,13.8,5.4,24.1,24.7,100.4


In [ ]:
# load correlations from previous colab (https://colab.research.google.com/drive/1RX1BMSk00eO_0e4hs-vDf2v5OfMT1CJq)

corr = pd.read_csv(os.path.join(BASE_DIR, "corr.csv"))

corr

,stat,correlation_with_win_pct
0,NRtg,9.512477e-01
1,MOV,9.510228e-01
2,SRS,9.472328e-01
3,SOS,-6.367829e-01
4,ORtg,6.278744e-01
...,...,...
164,opp_percent_FGA_16.3P,3.292808e-03
165,MP,-4.090350e-04
166,Year,2.458107e-15
167,G,4.881469e-16


In [ ]:

corr["stat"].values

'''
Top Stats to use:
1. Points per 100 (primary creator)
2. FGM (primary creator)
3. Opponent FG% (interior defender)
4. TS% (??)
5. Opponent eFG
6. % of shots that are 2 pointers (??)
7. % of opponent shots that are 2 pointers (interior defender)
8. Assist per 100 (support player)
9. Defensive rebound per 100 (interior defender)
10. opponent 3% (??)
11. Total Reb per 100 (interior defender)
12. Opponent assist per 100 (??)
'''

features = [
    "PTS_per_100",
    "FG.",
    "opp_FG%_per_game",
    "TS.",
    "Opp_eFG",
    "percent_2FGM",
    "opp_percent_2FGM",
    "AST_per_100",
    "DRB_per_100",
    "opp_3FG%_per_game",
    "TRB_per_100",
    "opp_AST_per_100"
]

# drop all rows not in features
corr_filtered = corr[corr["stat"].isin(features)]

corr_filtered = corr_filtered.copy()
corr_filtered["weight"] = corr_filtered["correlation_with_win_pct"].abs() / corr_filtered["correlation_with_win_pct"].abs().sum()

In [ ]:
corr_filtered

,stat,correlation_with_win_pct,weight
5,PTS_per_100,0.627874,0.107018
6,FG.,0.580455,0.098936
8,opp_FG%_per_game,-0.572521,0.097583
12,TS.,0.562698,0.095909
15,Opp_eFG,-0.495205,0.084405
16,percent_2FGM,0.475121,0.080982
17,opp_percent_2FGM,0.475121,0.080982
20,AST_per_100,0.456423,0.077795
22,DRB_per_100,0.438324,0.074710
26,opp_3FG%_per_game,-0.416483,0.070988


In [ ]:
# filter for 2024 data only
wnba_2024 = wnba_team_data[wnba_team_data["Year"] == 2024].copy()

# +1 means higher is better, -1 means lower is better
corr_filtered["direction"] = np.sign(corr_filtered["correlation_with_win_pct"]).replace(0, 1)


# normalize all stats
X = wnba_2024.apply(pd.to_numeric, errors="coerce")

mu = X.mean(axis=0)
sigma = X.std(axis=0, ddof=0).replace(0, np.nan)  # avoid divide-by-zero
Z = (X - mu) / sigma


# get deficit scores
direction_map = corr_filtered.set_index("stat")["direction"].to_dict()
weight_map = corr_filtered.set_index("stat")["weight"].to_dict()

deficit_scores = pd.DataFrame(index=wnba_2024.index)

for stat in corr_filtered["stat"]:
    deficit_scores[stat] = (-Z[stat]) * direction_map[stat]

# get weighted deficits
weighted_deficit_scores = deficit_scores.copy()
for stat in corr_filtered["stat"]:
    weighted_deficit_scores[stat] = weighted_deficit_scores[stat] * weight_map[stat]


# pivot tables
needs_long = (
    weighted_deficit_scores.assign(**{"Team": wnba_2024["Team"].values})
      .melt(id_vars=["Team"], var_name="stat", value_name="weighted_deficit")
      .sort_values(["Team", "weighted_deficit"], ascending=[True, False])
)

deficits_long = (
    deficit_scores.assign(**{"Team": wnba_2024["Team"].values})
     .melt(id_vars=["Team"], var_name="stat", value_name="deficit")
)

# Merge weighted and unweighted
needs_long = needs_long.merge(deficits_long, on=["Team", "stat"], how="left")

# add direction of stat weight to table
needs_long["weight"] = needs_long["stat"].map(weight_map)
needs_long["direction"] = needs_long["stat"].map(direction_map)

# Select the top 5 highest-priority needs for each team
top5_needs = needs_long.groupby("Team").head(5).reset_index(drop=True)
top5_needs



,Team,stat,weighted_deficit,deficit,weight,direction
0,Atlanta Dream,FG.,0.214575,2.168832,0.098936,1.0
1,Atlanta Dream,TS.,0.153596,1.601478,0.095909,1.0
2,Atlanta Dream,percent_2FGM,0.143513,1.772156,0.080982,1.0
3,Atlanta Dream,opp_percent_2FGM,0.143513,1.772156,0.080982,1.0
4,Atlanta Dream,PTS_per_100,0.136960,1.279780,0.107018,1.0
5,Chicago Sky,TS.,0.179864,1.875365,0.095909,1.0
6,Chicago Sky,percent_2FGM,0.140075,1.729709,0.080982,1.0
7,Chicago Sky,opp_percent_2FGM,0.140075,1.729709,0.080982,1.0
8,Chicago Sky,PTS_per_100,0.133899,1.251182,0.107018,1.0
9,Chicago Sky,FG.,0.114440,1.156710,0.098936,1.0
